# Game-Conditioned Kandinsky — Step 11: Paired Data Contract

This notebook prepares the data contract for a future Game→Audio adapter pilot. It does **not** download media, train a model, or claim any sample is cleared for commercial training.

The pilot is deliberately narrow: one event family (impact/collision), five-second synchronized video and target audio, engine-native event data preserved alongside a canonical representation, and evidence recorded for every media asset. Keep the text prompt fixed and generic during controlled comparisons so event details are not hidden in the prompt.

## 1. Create a clean workspace

The generated JSONL file is a blank template. You will later fill it only with recordings or assets whose rights have been reviewed.

In [ ]:
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

WORK_DIR = Path("/content/game_audio_data_contract")
WORK_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = WORK_DIR / "paired_samples.jsonl"
print("Workspace:", WORK_DIR)
print("Manifest template:", MANIFEST_PATH)

## 2. Define the engine-neutral event record

The canonical fields use SI units and seconds relative to the five-second clip. Optional measurements stay `null` when the engine does not expose them; **unknown is not zero**. Preserve the raw engine event so each engine integration can be audited and remapped later.

Each video and audio asset carries its own rights record. A dataset-level license label is not enough if underlying game footage, sound, or other third-party material has separate rights.

In [ ]:
def asset_template(relative_path):
    return {
        "path": relative_path,
        "sha256": None,
        "duration_s": None,
        "rights": {
            "rights_holder": None,
            "license_id": None,
            "license_text_url": None,
            "commercial_use_allowed": False,
            "commercial_model_training_allowed": False,
            "attribution_required": None,
            "attribution_text": None,
            "evidence_path_or_url": None,
            "review_status": "unverified"
        }
    }

TEMPLATE_RECORD = {
    "schema_version": "game-audio-pair-v0.1",
    "sample_id": "pilot_impact_0001",
    "capture_session_id": "replace_with_session_id",
    "split": "train",
    "clip_duration_s": 5.0,
    "text_prompt": "Audio that matches the provided gameplay video.",
    "prompt_policy": "fixed_generic_for_controlled_comparison",
    "video": asset_template("media/video/pilot_impact_0001.mp4"),
    "target_audio": asset_template("media/audio/pilot_impact_0001.wav"),
    "engine": {
        "name": None,
        "version": None,
        "game_or_scene_id": None,
        "raw_event_name": None,
        "raw_event_payload": {}
    },
    "events": [
        {
            "event_id": "event_001",
            "event_type": "impact",
            "event_time_s": 2.1,
            "audio_onset_s": None,
            "alignment_confidence": None,
            "material_a": None,
            "material_b": None,
            "object_mass_kg": None,
            "object_size_m": None,
            "impact_speed_m_s": None,
            "impulse_n_s": None,
            "distance_m": None,
            "reverb_rt60_s": None
        }
    ],
    "recorded_at_utc": datetime.now(timezone.utc).isoformat()
}

MANIFEST_PATH.write_text(
    json.dumps(TEMPLATE_RECORD, ensure_ascii=False) + "\n",
    encoding="utf-8"
)
print(json.dumps(TEMPLATE_RECORD, ensure_ascii=False, indent=2))
print("\nWrote template:", MANIFEST_PATH)

In [ ]:
import subprocess

def inspect_media_file(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    if not shutil.which("ffprobe"):
        raise RuntimeError("ffprobe is unavailable; use a Colab runtime with FFmpeg installed.")
    result = subprocess.run([
        "ffprobe", "-v", "error", "-show_entries",
        "format=duration:stream=codec_type,codec_name,sample_rate,channels,width,height,avg_frame_rate",
        "-of", "json", str(path)
    ], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(result.stderr[-2000:])
    probe = json.loads(result.stdout)
    digest = hashlib.sha256()
    with path.open("rb") as media_file:
        for chunk in iter(lambda: media_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return {
        "sha256": digest.hexdigest(),
        "duration_s": float(probe["format"]["duration"]),
        "measured_streams": probe.get("streams", [])
    }

def refresh_asset_metadata(record, asset_name):
    asset = record[asset_name]
    measured = inspect_media_file(WORK_DIR / asset["path"])
    asset.update(measured)
    return measured

print("Media inspector ready. Run it after the paired video and WAV have been placed under WORK_DIR.")

## 3. Check rights evidence and temporal alignment

This validator is intentionally conservative. It checks fields, media presence, clip duration, and event timestamps; it cannot decide whether a license legally covers model training. A human must review the license text and preserve the evidence. Records with unverified rights must not enter a training split.

In [ ]:
def validate_record(record, root=WORK_DIR, require_files=True):
    errors = []
    if record.get("schema_version") != "game-audio-pair-v0.1":
        errors.append("Unsupported or missing schema_version")
    duration = record.get("clip_duration_s")
    if duration != 5.0:
        errors.append("This Kandinsky pilot expects exactly 5.0-second paired clips")
    if not record.get("capture_session_id") or record["capture_session_id"].startswith("replace_"):
        errors.append("Set capture_session_id; split by session to avoid leakage")
    if not record.get("text_prompt") or record.get("prompt_policy") != "fixed_generic_for_controlled_comparison":
        errors.append("Use and record one fixed generic prompt for the controlled pilot")

    for asset_name in ("video", "target_audio"):
        asset = record.get(asset_name) or {}
        path = asset.get("path")
        if not path:
            errors.append(f"{asset_name}: missing path")
        elif require_files and not (root / path).is_file():
            errors.append(f"{asset_name}: file not found: {path}")
        measured_duration = asset.get("duration_s")
        if measured_duration is None or abs(measured_duration - 5.0) > 0.05:
            errors.append(f"{asset_name}: measured duration must be within 50 ms of 5.0 seconds")
        if not asset.get("sha256"):
            errors.append(f"{asset_name}: calculate and record a SHA-256 hash")
        rights = asset.get("rights") or {}
        required_rights = ["rights_holder", "license_id", "license_text_url", "evidence_path_or_url"]
        for key in required_rights:
            if not rights.get(key):
                errors.append(f"{asset_name}: missing rights.{key}")
        if rights.get("commercial_use_allowed") is not True:
            errors.append(f"{asset_name}: commercial use is not verified as allowed")
        if rights.get("commercial_model_training_allowed") is not True:
            errors.append(f"{asset_name}: commercial model training is not verified as allowed")
        if rights.get("review_status") != "verified":
            errors.append(f"{asset_name}: rights.review_status must be 'verified'")

    video_duration = (record.get("video") or {}).get("duration_s")
    audio_duration = (record.get("target_audio") or {}).get("duration_s")
    if video_duration is not None and audio_duration is not None and abs(video_duration - audio_duration) > 0.05:
        errors.append("Video and target audio durations differ by more than 50 ms")

    events = record.get("events") or []
    if not events:
        errors.append("At least one event is required")
    for index, event in enumerate(events):
        t = event.get("event_time_s")
        if t is None or not (0.0 <= t < 5.0):
            errors.append(f"events[{index}]: event_time_s must be inside [0, 5)")
        onset = event.get("audio_onset_s")
        if onset is None or not (0.0 <= onset < 5.0):
            errors.append(f"events[{index}]: measure audio_onset_s inside [0, 5)")
        if not event.get("event_type"):
            errors.append(f"events[{index}]: missing event_type")
    return errors

errors = validate_record(TEMPLATE_RECORD, require_files=False)
print("Template status:", "READY" if not errors else "NOT TRAINING READY")
print("Expected until real media and rights evidence are added:")
for error in errors:
    print(" -", error)

## 4. Dataset rules for the first pilot

- Keep the first pilot to one event family, such as a single impact/collision. Do not mix footsteps, doors, ambience, and impacts yet.
- Each row must pair the exact five-second video, target audio, and time-stamped event record from the same capture.
- Keep the text prompt fixed and generic across the controlled pilot. Do not encode material, intensity, or distance in the prompt.
- Store physical values in SI units. Record which values are missing; never silently replace missing telemetry with zero.
- Split by capture session or source scene, not by adjacent clips from the same take.
- Hash the media and keep a rights evidence URL or document for each audio/video asset. Do not treat a public download or a dataset's top-level license as proof by itself.
- This notebook does not start training. Training begins only after real pairs pass the rights and alignment review.

## What comes after this notebook

Once the paired media and license records exist, the next step is to import the manifest, verify actual durations/hashes and listen to aligned examples, then decide whether the pilot has enough coverage to train a small adapter. We will not train from the blank template.